# Outlier Mitigation

HDBSCAN assigns a `-1` label to points that do not fall within any dense region. This notebook demonstrates two approaches to recovering useful signal from that cluster:

1. **Soft clustering** — uses HDBSCAN membership vectors to reassign -1 points to fringe clusters
2. **K-means** — applies k-means directly to the -1 cluster as a complementary view

Dataset: 20 Newsgroups (5 categories, ~2 500 documents). No external data required.

See [`docs/outliers/`](https://github.com/ay94/topic-modeling-recipes/tree/main/docs/outliers) for the full methodology.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from collections import Counter

from sklearn.datasets import fetch_20newsgroups
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer

from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

from multilingual_topic.outlier_mitigation import StaticReducer, StaticClusterer, SoftReclusterer

## 1. Load data

In [ ]:
CATEGORIES = [
    'sci.space',
    'rec.sport.hockey',
    'talk.politics.guns',
    'comp.graphics',
    'soc.religion.christian',
]

raw = fetch_20newsgroups(
    subset='all',
    categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'),
)

docs = [d.strip() for d in raw.data]
# Drop very short documents
docs = [d for d in docs if len(d.split()) >= 10]
print(f"Documents: {len(docs)}")

## 2. Embed

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedding_model.encode(docs, show_progress_bar=True)
print(f"Embeddings shape: {embeddings.shape}")

## 3. UMAP + HDBSCAN

In [ ]:
# 2D UMAP for visualisation only
umap_2d = UMAP(n_neighbors=15, n_components=2, min_dist=0.0, random_state=1, metric='cosine')
coords_2d = umap_2d.fit_transform(embeddings)

# 5D UMAP for clustering (standard BERTopic setup)
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, random_state=1, metric='cosine')
umap_embeddings = umap_model.fit_transform(embeddings)

hdbscan_model = HDBSCAN(min_cluster_size=30, metric='euclidean', prediction_data=True)
hdbscan_labels = hdbscan_model.fit_predict(umap_embeddings)

topic_counts = pd.DataFrame(Counter(hdbscan_labels).items(), columns=['topic', 'count'])
topic_counts['pct'] = (topic_counts['count'] / topic_counts['count'].sum() * 100).round(1)
print(topic_counts.sort_values('count', ascending=False).to_string(index=False))

In [ ]:
# Visualise UMAP coloured by HDBSCAN labels
plot_df = pd.DataFrame(coords_2d, columns=['x', 'y'])
plot_df['topic'] = [f'topic-{t}' for t in hdbscan_labels]

fig = px.scatter(plot_df, x='x', y='y', color='topic', title='UMAP — HDBSCAN topics')
fig.show()

## 4. BERTopic

We pass the pre-computed UMAP embeddings and HDBSCAN labels via `StaticReducer` and `StaticClusterer` to avoid rerunning dimensionality reduction inside BERTopic. Setting `calculate_probabilities=True` produces the membership vectors needed for soft clustering.

In [ ]:
vectorizer = CountVectorizer(stop_words='english', min_df=2)

topic_model = BERTopic(
    umap_model=StaticReducer(umap_embeddings),
    hdbscan_model=StaticClusterer(hdbscan_labels),
    vectorizer_model=vectorizer,
    calculate_probabilities=True,
    verbose=True,
)

topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)
topic_info = topic_model.get_topic_info()
topic_info

In [ ]:
n_outlier = sum(t == -1 for t in topics)
print(f"-1 cluster: {n_outlier} / {len(topics)} docs ({n_outlier/len(topics)*100:.1f}%)")

## 5. Soft clustering

The `SoftReclusterer` uses the HDBSCAN membership vector for each -1 point to assign it to one or more fringe clusters. Points whose membership scores do not meet the thresholds remain as true outliers.

Key parameters:
- `method='ratio'` — threshold = `max_score × threshold_ratio` per point
- `threshold_ratio=0.5` — any cluster scoring ≥ 50% of the max qualifies
- `min_membership_score=0.05` — ignore clusters with negligible scores
- `max_core_clusters=3` — a point can be fringe of at most 3 core topics
- `min_fringe_cluster_size=5` — groups smaller than 5 docs are treated as outliers

In [ ]:
soft = SoftReclusterer(
    original_labels=topics,
    membership_vectors=probs,
    method='ratio',
    threshold_ratio=0.5,
    min_membership_score=0.05,
    max_core_clusters=3,
    min_fringe_cluster_size=5,
)

fringe_labels = soft.fit_predict(umap_embeddings)
fringe_names = [soft.build_display_name(f) for f in fringe_labels]
soft.summary()

In [ ]:
# Visualise fringe assignments
plot_df['fringe'] = fringe_names

fig = px.scatter(
    plot_df, x='x', y='y', color='fringe',
    title='UMAP — soft clustering fringe assignments',
    hover_data=['topic'],
)
fig.show()

In [ ]:
# Fringe linkage network — which fringe groups straddle which core topics
G = soft.fringe_linkage_network()
print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
for u, v, w in G.edges(data='weight'):
    print(f"  {soft.build_display_name(u):30s} <-> {soft.build_display_name(v):30s}  weight={w}")

## 6. K-means comparison

K-means is applied to the -1 cluster only, with k equal to the number of topics found by HDBSCAN. Since the -1 cluster mirrors the shape of the full data (see Figure 1 in the outlier docs), using the same k provides a direct structural comparison.

In [ ]:
n_core_topics = len([t for t in topic_info['Topic'] if t != -1])
outlier_mask = np.array(topics) == -1

kmeans = KMeans(n_clusters=n_core_topics, random_state=1, n_init='auto')
k_labels = kmeans.fit_predict(embeddings[outlier_mask])

print(f"K-means applied to {outlier_mask.sum()} outlier docs, k={n_core_topics}")
print(pd.Series(k_labels).value_counts().rename('count').to_string())

In [ ]:
# Visualise k-means on the outlier subset
kmeans_df = plot_df[outlier_mask].copy()
kmeans_df['k_topic'] = [f'k-{l}' for l in k_labels]

fig = px.scatter(
    kmeans_df, x='x', y='y', color='k_topic',
    title='UMAP — k-means on -1 cluster',
)
fig.show()

## 7. Sample per fringe cluster

Draw a 10% sample from each fringe group for manual annotation.

In [ ]:
result_df = pd.DataFrame({'text': docs, 'topic': topics, 'fringe': fringe_names})

samples = []
for fringe_name, group in result_df.groupby('fringe'):
    n = max(1, int(len(group) * 0.1))
    samples.append(group.sample(n=n, random_state=1))

sample_df = pd.concat(samples).sort_values('fringe')
print(f"Sample: {len(sample_df)} docs across {result_df['fringe'].nunique()} fringe groups")
sample_df.head(10)